# Fama-French五因子归因分析
**基金**: 中欧周期优选混合 (019888)
**分析区间**: 2022-01-01 ~ 2024-12-31

---

In [ ]:
import sys
import os

project_root = os.path.dirname(os.path.abspath('.'))
sys.path.insert(0, os.path.join(project_root, 'source'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

print('依赖库加载完成')

## 1. 数据加载

In [ ]:
from data_loader import FundDataLoader, FactorDataLoader
from factor import FamaFrenchFactorBuilder
from backtest import FamaFrenchAttribution
from plot import FamaFrenchPlotter

FUND_CODE = '019888'
FUND_NAME = '中欧周期优选混合'
START_DATE = '2022-01-01'
END_DATE = '2024-12-31'

OUTPUT_DIR = r'C:\Users\chenh\.qclaw\workspace\fama-factor-attribution\output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'基金: {FUND_NAME} ({FUND_CODE})')
print(f'分析区间: {START_DATE} ~ {END_DATE}')

## 2. 加载基金净值数据

In [ ]:
fund_loader = FundDataLoader()
fund_data = fund_loader.get_fund_nav(FUND_CODE, START_DATE, END_DATE, 'monthly')
print(f'基金净值记录: {len(fund_data)} 条')
print(fund_data.tail())

## 3. 构建Fama-French五因子

In [ ]:
factor_builder = FamaFrenchFactorBuilder()
factors = factor_builder.build_five_factors(START_DATE, END_DATE)
print(f'因子数据: {factors.shape}')
print(factors.head())

## 4. 运行回归归因

In [ ]:
attribution = FamaFrenchAttribution()
regression_results = attribution.run_regression(fund_data, factors)
print('回归完成')

## 5. 因子贡献分解

In [ ]:
contrib_df = attribution.calculate_factor_contribution(factors)
print('因子贡献分解完成')

## 6. 滚动回归

In [ ]:
rolling_df = attribution.run_rolling_regression(fund_data, factors, window=12)
print('滚动回归完成')

## 7. 可视化

In [ ]:
plotter = FamaFrenchPlotter()

# 因子暴露柱状图
fig = plotter.plot_factor_exposure(regression_results, FUND_NAME)
fig.savefig(os.path.join(OUTPUT_DIR, 'fama_factor_exposure.png'), bbox_inches='tight')
plt.close()
print('因子暴露图已保存')

# 收益分解饼图
fig = plotter.plot_return_decomposition(contrib_df, FUND_NAME)
fig.savefig(os.path.join(OUTPUT_DIR, 'fama_return_decomposition.png'), bbox_inches='tight')
plt.close()
print('收益分解图已保存')

# 滚动暴露图
if len(rolling_df) > 0:
    fig = plotter.plot_rolling_exposure(rolling_df, FUND_NAME)
    fig.savefig(os.path.join(OUTPUT_DIR, 'fama_rolling_exposure.png'), bbox_inches='tight')
    plt.close()
    rolling_df.to_csv(os.path.join(OUTPUT_DIR, 'fama_rolling_exposure.csv'), encoding='utf-8-sig')
    print('滚动暴露图已保存')

## 8. 保存结果

In [ ]:
# 保存因子数据
factors.to_csv(os.path.join(OUTPUT_DIR, 'fama_five_factors.csv'), encoding='utf-8-sig')

# 保存回归结果
results_df = pd.DataFrame({
    '因子': ['Alpha(年化)', 'R_M(市场)', 'SMB(市值)', 'HML(价值)', 'RMW(盈利)', 'CMA(投资)'],
    '系数': [
        regression_results['alpha'] * 12,
        regression_results['params']['R_M'],
        regression_results['params']['SMB'],
        regression_results['params']['HML'],
        regression_results['params']['RMW'],
        regression_results['params']['CMA']
    ],
    't值': [
        regression_results['alpha_t'],
        regression_results['t_values']['R_M'],
        regression_results['t_values']['SMB'],
        regression_results['t_values']['HML'],
        regression_results['t_values']['RMW'],
        regression_results['t_values']['CMA']
    ],
    'P值': [
        regression_results['alpha_p'],
        regression_results['p_values']['R_M'],
        regression_results['p_values']['SMB'],
        regression_results['p_values']['HML'],
        regression_results['p_values']['RMW'],
        regression_results['p_values']['CMA']
    ]
})
results_df.to_csv(os.path.join(OUTPUT_DIR, 'fama_attribution_results.csv'), encoding='utf-8-sig', index=False)

# 保存因子贡献
contrib_df.to_csv(os.path.join(OUTPUT_DIR, 'fama_factor_contribution.csv'), encoding='utf-8-sig', index=False)

print('所有结果已保存到:', OUTPUT_DIR)
print('\n分析完成!')

## 9. 结果解读

In [ ]:
# 归因摘要
summary = attribution.get_attribution_summary()
print('=' * 60)
print(f'Fama-French五因子归因分析结果 - {FUND_NAME}')
print('=' * 60)
print(f"模型解释力度(R2): {summary['模型解释力度']:.2%}")
print(f"Alpha(年化): {summary['Alpha年化']:.2%} {'(显著)' if summary['Alpha显著'] else '(不显著)'}")
print(f"市场Beta: {summary['市场Beta']:.4f}")
print(f"显著因子数: {summary['显著因子数']}")
print(f"显著因子: {summary['显著因子']}")
print('=' * 60)